In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, AutoModelForCausalLM
import torch

2025-12-30 01:19:17.571391: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767057557.779545      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767057557.838805      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767057558.329066      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767057558.329122      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767057558.329128      55 computation_placer.cc:177] computation placer alr

In [3]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')

In [4]:
model_id = 'VietAI/vit5-base'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

In [5]:
dataset_id = 'vohuutridung/3190-data'
train_ds = load_dataset(dataset_id, split='train')
valid_ds = load_dataset(dataset_id, split='validation')

README.md:   0%|          | 0.00/306 [00:00<?, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/597 [00:00<?, ? examples/s]

In [6]:
MAX_INPUT_LEN = 256
MAX_OUTPUT_LEN = 256

def preprocess(example):
    model_inputs = tokenizer(
        example["text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            str(example["labels"]),
            max_length=MAX_OUTPUT_LEN,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = train_ds.map(preprocess, batched=False)
valid_ds = valid_ds.map(preprocess, batched=False)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./vit5_absa",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,

    eval_steps=200,
    eval_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    save_strategy="steps",
    logging_steps=100,
    
    report_to="tensorboard",
    predict_with_generate=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
)

In [8]:
trainer.train()

Step,Training Loss,Validation Loss
200,0.140700,0.104663
400,0.105600,0.083926
600,0.088000,0.073617
800,0.071300,0.068434
1000,0.069400,0.066447
1200,0.067500,0.062473
1400,0.055000,0.063274
1600,0.055000,0.061321
1800,0.051700,0.059410
2000,0.045100,0.060134


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3125, training_loss=0.09586781299591064, metrics={'train_runtime': 3419.5354, 'train_samples_per_second': 14.622, 'train_steps_per_second': 0.914, 'total_flos': 1.5223947264e+16, 'train_loss': 0.09586781299591064, 'epoch': 5.0})

In [9]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(HF_TOKEN)


REPO_ID = 'vohuutridung/vit5-base-absa-v2'
trainer.model.push_to_hub(REPO_ID)
trainer.processing_class.push_to_hub(REPO_ID)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/vohuutridung/vit5-base-absa-v2/commit/4eb559e737a5b3ed098968b964a8f6ea5450a805', commit_message='Upload tokenizer', commit_description='', oid='4eb559e737a5b3ed098968b964a8f6ea5450a805', pr_url=None, repo_url=RepoUrl('https://huggingface.co/vohuutridung/vit5-base-absa-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='vohuutridung/vit5-base-absa-v2'), pr_revision=None, pr_num=None)

In [10]:
def infer(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(infer("Pin máy dùng được lâu"))


[['Pin máy', 'PIN', 'TÍCH_CỰC', 'dùng được lâu']]
